# AI Trading System Phase 2: Hyperparameter Tuning with Optuna
**Objective:** Optimize hyperparameters for LightGBM and RandomForest to maximize Macro Precision using Optuna and TimeSeriesSplit.

In [ ]:
import numpy as np
import pandas as pd
import ta
import optuna
import warnings
from vnstock import Vnstock

# Models
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import precision_score

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Data Preparation
Fetch 5-minute historical data for `VN30F1M` or fallback to synthetic Random Walk OHLCV data. Calculate indicators and generate labels.

In [ ]:
# Try to fetch real data from vnstock, otherwise use synthetic data
try:
    print("Attempting to fetch real data using vnstock...")
    end_date = pd.Timestamp.today().strftime('%Y-%m-%d')
    start_date = (pd.Timestamp.today() - pd.DateOffset(months=6)).strftime('%Y-%m-%d')
    stock = Vnstock().derivative(symbol='VN30F1M', source='VCI')
    df = stock.quote.history(start=start_date, end=end_date, resolution='5')
    
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        df.set_index('time', inplace=True)
    elif 'tradingDate' in df.columns:
        df['tradingDate'] = pd.to_datetime(df['tradingDate'])
        df.set_index('tradingDate', inplace=True)
        
    df.columns = [c.lower() for c in df.columns]
    print(f"Successfully fetched real data: {df.shape}")
except Exception as e:
    print(f"Failed to fetch real data: {e}")
    print("Generating synthetic Random Walk OHLCV data instead...")
    np.random.seed(42)
    periods = 5000
    dates = pd.date_range(start='2025-01-01', periods=periods, freq='5min')
    
    # Random walk
    price_changes = np.random.normal(0, 1.5, periods)
    close = 1200 + np.cumsum(price_changes)
    
    df = pd.DataFrame({
        'open': close + np.random.uniform(-0.5, 0.5, periods),
        'high': close + np.random.uniform(0, 2.0, periods),
        'low': close - np.random.uniform(0, 2.0, periods),
        'close': close,
        'volume': np.random.randint(100, 5000, periods)
    }, index=dates)
    print(f"Generated synthetic data: {df.shape}")

# Calculate Indicators
df['rsi_14'] = ta.momentum.RSIIndicator(close=df['close'], window=14).rsi()

macd = ta.trend.MACD(close=df['close'])
df['macd'] = macd.macd()
df['macd_signal'] = macd.macd_signal()
df['macd_diff'] = macd.macd_diff()

bb = ta.volatility.BollingerBands(close=df['close'], window=20, window_dev=2)
df['bb_mavg'] = bb.bollinger_mavg()
df['bb_high'] = bb.bollinger_hband()
df['bb_low'] = bb.bollinger_lband()

df['vwap'] = ta.volume.VolumeWeightedAveragePrice(
    high=df['high'], low=df['low'], close=df['close'], volume=df['volume']
).volume_weighted_average_price()

df.dropna(inplace=True)

# Generate Labels
THRESHOLD = 1.0
df['next_close'] = df['close'].shift(-1)
df['price_change'] = df['next_close'] - df['close']
df.dropna(inplace=True)

def assign_label(change):
    if change > THRESHOLD:
        return 1
    elif change < -THRESHOLD:
        return -1
    else:
        return 0

df['target'] = df['price_change'].apply(assign_label)

# Prepare Features & Target
features = [
    'open', 'high', 'low', 'close', 'volume', 
    'rsi_14', 'macd', 'macd_signal', 'macd_diff', 
    'bb_mavg', 'bb_high', 'bb_low', 'vwap'
]
X = df[features]
y = df['target']
print(f"Final dataset shape: X={X.shape}, y={y.shape}")
print("Class Distribution:")
print(y.value_counts(normalize=True))

## 2. Optuna Tuning: LightGBM
- Search Space: `num_leaves` (10-50), `learning_rate` (0.01-0.1), `max_depth` (3-8), `min_child_samples` (10-50).
- Trials: 20
- Validation: TimeSeriesSplit(n_splits=3)

In [ ]:
def lgbm_objective(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 10, 50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'random_state': 42,
        'verbose': -1
    }
    
    model = LGBMClassifier(**params)
    tscv = TimeSeriesSplit(n_splits=3)
    precisions = []
    
    for train_index, test_index in tscv.split(X):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Maximize Macro Precision
        precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
        precisions.append(precision)
        
    return np.mean(precisions)

print("Starting LightGBM Optuna Optimization (20 trials)...")
lgbm_study = optuna.create_study(direction='maximize')
lgbm_study.optimize(lgbm_objective, n_trials=20)

print(f"Best LightGBM Macro Precision: {lgbm_study.best_value:.4f}")
print("Best LightGBM Params:")
for k, v in lgbm_study.best_params.items():
    print(f"  {k}: {v}")

## 3. Optuna Tuning: RandomForest
- Search Space: `n_estimators` (50-200), `max_depth` (3-10), `min_samples_split` (2-10).
- Trials: 20
- Validation: TimeSeriesSplit(n_splits=3)

In [ ]:
def rf_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'random_state': 42,
        'n_jobs': -1  # Speed up training
    }
    
    model = RandomForestClassifier(**params)
    tscv = TimeSeriesSplit(n_splits=3)
    precisions = []
    
    for train_index, test_index in tscv.split(X):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Maximize Macro Precision
        precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
        precisions.append(precision)
        
    return np.mean(precisions)

print("Starting RandomForest Optuna Optimization (20 trials)...")
rf_study = optuna.create_study(direction='maximize')
rf_study.optimize(rf_objective, n_trials=20)

print(f"Best RandomForest Macro Precision: {rf_study.best_value:.4f}")
print("Best RandomForest Params:")
for k, v in rf_study.best_params.items():
    print(f"  {k}: {v}")